# 24-009 Runx3OE Numbers Analysis

This notebook details counts analysis for 24-009 for Figure 1.

## Initialize Environment

In [1]:
# Import packages
library(fcexpr)
library(dplyr)
library(tidyr)
library(ggplot2)
library(ggpubr)
library(ggsci)
library(ggprism)

setwd("/home/dalbao/AlbaoRunx3Manuscript/flow/Fig01")

# Import Flow processing functions
source("/home/dalbao/AlbaoRunx3Manuscript/flow/scripts/flowProcessing.R")


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘rstatix’


The following object is masked from ‘package:stats’:

    filter



Attaching package: ‘purrr’


The following objects are masked from ‘package:rlang’:

    %@%, flatten, flatten_chr, flatten_dbl, flatten_int, flatten_lgl,
    flatten_raw, invoke, splice




## Load Data

In [2]:
# Load data
workspace <- load_workspace_cached(wsp_file = "2024-009-260807-AutoGate.wsp", rds_file = "2024-009-260807-AutoGate.rds")
counts    <- workspace[["counts"]]
stats     <- workspace[["stats"]]

# Make backup
.workspace <- workspace

# Import total numbers
numbers <- read.csv("2024-009_counts.csv")

## Clean and Process Flow Data

First pre-process into percent of subsets of interest:

In [3]:
# Isolate Populations of Interest
subsets <- counts %>%
  filter(grepl("Runx3|mock", Population))

# Population contains metadata for two columns
# Transduction and Subset, separated by a /
# We can split these into separate columns
subsets <- subsets %>%
  separate(Population, c("Transduction", "Subset", "Kaech_in_Gerlach"), sep = "/")
# Replace NA in Subset with "Total"
subsets$Subset[is.na(subsets$Subset)] <- "Total"

# Remove observations "Concat" from FlowJoGroup
subsets <- subsets %>%
  filter(FlowJoGroup != "Concat")

# FlowJoGroup ontains metadata for two columns
# Genotype and Day separated by a -
# We can split these into separate columns
subsets <- subsets %>%
  separate(FlowJoGroup, c("Genotype", "Day"), sep = "-")

# Reorganize subsets to relevant information
subsets <- subsets %>%
  select(FileName, Genotype, Day, Transduction, Subset, Kaech_in_Gerlach, FractionOfParent)

# Convert FractionOfParent to Percent and Rename
subsets$FractionOfParent <- subsets$FractionOfParent * 100
subsets <- subsets %>%
  rename(Percent = FractionOfParent)

# Samples with filename D65_KO4.fcs, D65_WT4.fcs and D150_KO5.fcs are of bad quality and should be removed from the analysis. (see lab notebook)
# Write a script to remove obvservations containing D65_KO4.fcs, D65_WT4.fcs and D150_KO5.fcs from the analysis
subsets <- subsets %>%
  filter( FileName != "D65_KO4.fcs" &
          FileName != "D65_WT4.fcs" &
          FileName != "D150_KO5.fcs")

#Samples annotated as Day D6.5 are actually D8.0 and D8.0 and vice versa. (see lab notebook)
# Write a script to correct the Day annotation
subsets$Day <- gsub("D6.5", "temp", subsets$Day)
subsets$Day <- gsub("D8.0", "D6.5", subsets$Day)
subsets$Day <- gsub("temp", "D8.0", subsets$Day)

# Check subsets
head(subsets)

# Write to CSV
write.csv(subsets, "2024-009_percent_of_parents.csv", row.names = FALSE)

Warning message:
“Expected 3 pieces. Missing pieces filled with `NA` in 958 rows [1, 2, 3, 4, 5,
6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, ...].”
Warning message:
“Expected 2 pieces. Missing pieces filled with `NA` in 14 rows [515, 516, 1097,
1098, 1099, 1100, 1101, 1102, 1683, 1684, 1685, 1686, 1687, 1688].”


,FileName,Genotype,Day,Transduction,Subset,Kaech_in_Gerlach,Percent
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>
13,D150_KO1.fcs,KO,D15.0,mock,Total,NA,10.337503
14,D150_KO1.fcs,KO,D15.0,Runx3,Total,NA,89.568091
15,D150_KO1.fcs,KO,D15.0,mock,DP,NA,3.881279
16,D150_KO1.fcs,KO,D15.0,mock,EE,NA,13.013699
17,D150_KO1.fcs,KO,D15.0,mock,MP,NA,23.287671
18,D150_KO1.fcs,KO,D15.0,mock,Sel-Hi,NA,2.739726


Then process percent of parents.

In [4]:
# Isolate Populations of Interest
parents <- counts %>%
  filter(grepl("Live", Parent))
# Keep relevant clones
parents <- parents %>%
  filter(grepl("CD8|Transferred|Transduced|mock|Runx3", Population) &
         !(grepl("Sel|\\+", Population)))

# Remove observations "Concat" from FlowJoGroup
parents <- parents %>%
  filter(FlowJoGroup != "Concat")

# FlowJoGroup ontains metadata for two columns
# Genotype and Day separated by a -
# We can split these into separate columns
parents <- parents %>%
  separate(FlowJoGroup, c("Genotype", "Day"), sep = "-")

# Reorganize parents to relevant information
parents <- parents %>%
  select(FileName, Genotype, Day, Population, FractionOfParent)

# Samples with filename D65_KO4.fcs, D65_WT4.fcs and D150_KO5.fcs are of bad quality and should be removed from the analysis.
# Write a script to remove obvservations containing D65_KO4.fcs, D65_WT4.fcs and D150_KO5.fcs from the analysis
parents <- parents %>%
  filter( FileName != "D65_KO4.fcs" &
          FileName != "D65_WT4.fcs" &
          FileName != "D150_KO5.fcs")

#Samples annotated as Day D6.5 are actually D8.0 and D8.0 and vice versa.
# Write a script to correct the Day annotation
parents$Day <- gsub("D6.5", "temp", parents$Day)
parents$Day <- gsub("D8.0", "D6.5", parents$Day)
parents$Day <- gsub("temp", "D8.0", parents$Day)

# Spread parents by Population
parents <- spread(parents, key = Population, value = FractionOfParent)

# Check parents
head(parents)

,FileName,Genotype,Day,CD8,mock,mock/DP,mock/EE,mock/Lo,mock/MP,mock/Non-Term,⋯,Runx3/Tem/MP,Runx3/Tem/TE,Runx3/Term,Runx3/Tpm,Runx3/Tpm/DP,Runx3/Tpm/EE,Runx3/Tpm/MP,Runx3/Tpm/TE,Transduced,Transferred
,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,D150_KO1.fcs,KO,D15.0,0.2005492,0.10337503,0.03881279,0.13013699,NA,0.23287671,NA,⋯,0.11976048,0.4940120,NA,0.7507246,0.4018954,0.009126009,0.5777466,0.01123201,0.6967604,0.024891629
2,D150_KO2.fcs,KO,D15.0,0.2075756,0.19069767,0.02439024,0.17804878,NA,0.21463415,NA,⋯,0.07341772,0.6911392,NA,0.6084442,0.3355513,0.016159696,0.6292776,0.01901141,0.4924416,0.010926500
3,D150_KO3.fcs,KO,D15.0,0.2036167,0.05068337,0.01123596,0.16853933,NA,0.35955056,NA,⋯,0.10035842,0.6594982,NA,0.6793249,0.3398403,0.025732032,0.6149068,0.01952085,0.6050999,0.006755813
4,D150_KO4.fcs,KO,D15.0,0.1949606,0.23344948,0.00000000,0.07462687,NA,0.07462687,NA,⋯,0.12903226,0.6290323,NA,0.5844749,0.3593750,0.046875000,0.5937500,0.00000000,0.4361702,0.001620506
5,D150_WT1.fcs,WT,D15.0,0.2063426,0.16220971,0.09761388,0.00867679,NA,0.04338395,NA,⋯,0.08666667,0.4033333,NA,0.7766092,0.6245937,0.003791983,0.3569881,0.01462622,0.4368275,0.014134107
6,D150_WT2.fcs,WT,D15.0,0.1899135,0.13290559,0.18103448,0.01551724,NA,0.11896552,NA,⋯,0.06349206,0.5379189,NA,0.7128110,0.5246937,0.005569996,0.4567397,0.01299666,0.3710884,0.029387951


Then compute total frequencies:

In [5]:
# Isolate colnames in parents which contain "/"
# These are the populations of interest
populations <- colnames(parents)[grepl("mock|Runx3", colnames(parents))]

# populations can contain either one, two or three elements, separated by "/"
# Split this into a list with the following conditions:
# If length of elements in populations is 2, like "CD8/Transferred"
# Split like: "CD8", and "CD8/Transferred"
# If length of elements in populations
# is 3, like "CD8/Transferred/Transduced"
# Split like: "CD8", "CD8/Transferred", and "CD8/Transferred/Transduced"
split_population <- function(population) {
  # Split the population string by "/"
  elements <- strsplit(population, "/")[[1]]

  # Initialize the result vector
  result <- c()

  # Create cumulative substrings
  for (i in 1:length(elements)) {
    result <- c(result, paste(elements[1:i], collapse = "/"))
  }

  return(result)
}

# Split populations into a list
populations <- lapply(populations, split_population)

library(matrixStats)
# Compute total frequencies into a new df total_freq
# This will be done by multiplying the frequencies within
# the gating hierarchy to each column in populations
total_freq <- parents[1:3]

# Loop over each gating hierarchy
for (i in 1:length(populations)) {
  column <- populations[[i]][length(populations[[i]])]
  total_freq[, column] <-
    rowProds(
      as.matrix(
        parents[,
          c("CD8",
            "Transferred",
            "Transduced",
            populations[[i]]
          )
        ]
      )
    )
}
rm(i, column, populations)


Attaching package: ‘matrixStats’


The following object is masked from ‘package:dplyr’:

    count




## Process Counts

In [6]:
# Import total numbers
numbers <- read.csv("2024-009_counts.csv")
# Recreate Incorrect FileNames
numbers$FileNameIncorrect <- paste( "D",
                                    numbers$DayIncorrect,
                                    "_",
                                    numbers$Sample,
                                    ".fcs", sep = "")
# Rorder numbers$FileNameIncorrect to match total_freq$FileName
numbers <- numbers[match(total_freq$FileName, numbers$FileNameIncorrect), ]
# Create total_numbers df
total_numbers <- total_freq
# Isolate populations of interest
populations <- colnames(parents)[grepl("mock|Runx3", colnames(parents))]
# Calculate numbers by multiplying total_freq by numbers
for (pop in populations) {
  total_numbers[, pop] <- total_freq[, pop] * numbers$Total_Cells
}
rm(pop)

# Split total_numbers into two dfs by Genotype
total_numbers_WT <- total_numbers %>%
  filter(Genotype == "WT")
total_numbers_KO <- total_numbers %>%
  filter(Genotype == "KO")

# Make vectors of mock and Runx3 populations
pop_mock <- populations <- colnames(parents)[grepl("mock", colnames(parents))]
pop_Runx3 <- populations <- colnames(parents)[grepl("Runx3", colnames(parents))]

# Normalize total_numbers by dividing to normalization factors:angle
# FROM LAB NOTEBOK, EMPIRICALLY DETERMINED:angle
# KO - Runx3 - 0.678
# KO - mock - 0.598
# WT - Runx3 -  0.869
# WT - mock - 0.841
total_numbers_KO[, pop_mock] <- total_numbers_KO[, pop_mock] / 0.598
total_numbers_KO[, pop_Runx3] <- total_numbers_KO[, pop_Runx3] / 0.678
total_numbers_WT[, pop_mock] <- total_numbers_WT[, pop_mock] / 0.841
total_numbers_WT[, pop_Runx3] <- total_numbers_WT[, pop_Runx3] / 0.869

# Combine total_numbers_WT and total_numbers_KO
total_numbers_norm <- rbind(total_numbers_WT, total_numbers_KO)

# Cleanup
rm(pop_mock, pop_Runx3, total_numbers_WT, total_numbers_KO, populations)

# Process total_numbers_norm
# First gather
total_numbers_norm <- gather(total_numbers_norm, key = "Population", value = "Number", 4:ncol(total_numbers_norm))

# Population contains metadata for two columns
# Transduction and Subset, separated by a /
# We can split these into separate columns
total_numbers_norm <- total_numbers_norm %>%
  separate(Population, c("Transduction", "Subset", "Kaech_in_Gerlach"), sep = "/")

# Label Subset that is NA as Total
total_numbers_norm$Subset[is.na(total_numbers_norm$Subset)] <- "Total"

# Compatibility, rename Number as Percent


Warning message:
“Expected 3 pieces. Missing pieces filled with `NA` in 594 rows [1, 2, 3, 4, 5,
6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, ...].”


In [7]:
head(total_numbers_norm)

# Write to CSV
write.csv(total_numbers_norm, "2024-009_total_numbers.csv", row.names = FALSE)

,FileName,Genotype,Day,Transduction,Subset,Kaech_in_Gerlach,Number
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>
1,D150_WT1.fcs,WT,D15.0,mock,Total,NA,11819.83
2,D150_WT2.fcs,WT,D15.0,mock,Total,NA,20000.17
3,D150_WT3.fcs,WT,D15.0,mock,Total,NA,46756.83
4,D150_WT4.fcs,WT,D15.0,mock,Total,NA,25137.38
5,D150_WT5.fcs,WT,D15.0,mock,Total,NA,18216.95
6,D65_WT1.fcs,WT,D8.0,mock,Total,NA,2109185.68


In [8]:
mean_total_numbers_norm <- total_numbers_norm %>% # Remove column FileName
    select(-FileName) %>%
    group_by(Genotype, Day, Transduction, Subset, Kaech_in_Gerlach) %>%
    summarize(mean_Number = mean(Number),
                sd_Number = sd(Number),
                n = n(),
                se_Number = sd(Number) / sqrt(n))

# Save mean_total_numbers_norm to csv
write.csv(mean_total_numbers_norm, "2024-009_mean_total_numbers.csv",
          row.names = FALSE)

`summarise()` has grouped output by 'Genotype', 'Day', 'Transduction',
'Subset'. You can override using the `.groups` argument.


## Plot

In [9]:
# Make plot_data df
plot_data <- total_numbers_norm %>%
    # Subset to just isolate relevant populations
    filter( Subset %in% c("Total", "Tcm", "Tpm", "Tem"),
            is.na(Kaech_in_Gerlach),
            !(Genotype == "KO" & Transduction == "Runx3")
    ) %>% 
    # Remove column Kaech_in_Gerlach
    select(-Kaech_in_Gerlach) %>%
    # Make new column Treatment
    # If Transduction is mock and Genotype is KO, Treatment is KO
    # If Transduction is mock and Genotype is WT, Treatment is WT
    # If Transduction is Runx3 and Genotype is WT, Treatment is OE
    mutate(Treatment = case_when(
        Genotype == "KO" & Transduction == "mock" ~ "KO",
        Genotype == "WT" & Transduction == "mock" ~ "WT",
        Genotype == "WT" & Transduction == "Runx3" ~ "OE",
        TRUE ~ NA_character_
    )) %>%
    # Factorize Treatment with levels KO, WT, OE
    # Factorize Day with levels D6.5, D8.0, D15.0
    # Factorize Subset with levels Total, Tcm, Tpm, Tem
    mutate( Treatment = factor(Treatment, levels = c("KO", "WT", "OE")),
            Day = factor(Day, levels = c("D6.5", "D8.0", "D15.0")),
            Subset = factor(Subset, levels = c("Total", "Tcm", "Tpm", "Tem"))
    )


In [10]:
# First define a general theme
# Use theme_bw, base 12, not bold, and remove legend
theme <- theme_bw(      base_size = 14,
                        base_family = "sans"
                        ) +
    theme(  axis.text = element_text(size = 10, color = "black"),
            axis.title = element_text(size = 12, color = "black"),
            axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1, size = 12, color = "black"),
            axis.title.x = element_blank(),
            strip.text = element_text(size = 12, color = "black"),
            legend.position = "none",
            plot.title = element_text(size = 12, color = "black"),  # Set main title size
            panel.grid = element_blank())  # Remove gridlines


In [11]:
colors <- c("WT" = "#808180",
            "KO" = "#DC0000",
            "OE" = "#00A087")

# Compute BH-adjusted pairwise comparisons with plot_padj(), adjusting
# across all Day x Subset panels together (adjust_scope = "total"),
# replacing the unadjusted stat_compare_means() p-values used previously
stat_data <- plot_data %>%
    mutate(FacetKey = paste(Day, Subset, sep = "_"))

stat.test <- plot_padj(
    data            = stat_data,
    comparisons     = list(c("WT", "KO"), c("WT", "OE")),
    p_adjust_method = "BH",
    y_col           = "Number",
    x_col           = "Treatment",
    facet_col       = "FacetKey",
    adjust_scope    = "intrafacet",
    paired          = FALSE
) %>%
    separate(FacetKey, into = c("Day", "Subset"), sep = "_") %>%
    mutate( Day    = factor(Day, levels = c("D6.5", "D8.0", "D15.0")),
            Subset = factor(Subset, levels = c("Total", "Tcm", "Tpm", "Tem")))

# Export full comparison stats (p, p.adj, fold changes, effect sizes) for reporting
write.csv(stat.test, "2024-009-Fig01i.csv", row.names = FALSE)

# Build bracket labels showing fold change (mean(group2) / mean(group1)),
# with significance stars appended only when significant (kept short so
# labels fit on a 12-panel grid)
format_fc <- function(x) ifelse(abs(x) < 1, signif(x, 1), round(x, 1))
stat.test <- stat.test %>%
    mutate(
        fc_label = paste0(format_fc(foldmean), "x", ifelse(p.adj.signif == "ns", "", p.adj.signif)),
        # geom_bracket()'s y.position aesthetic bypasses scale_y_log10()'s data
        # transform, so it must be supplied already in log10 units, or the
        # panel's y-range gets corrupted (real data collapses to a sliver)
        y.position = log10(y.position)
    )

p <- ggplot(plot_data, aes(x = Treatment, y = Number, fill = Treatment)) +
    geom_boxplot(outliers = FALSE) +
    geom_point(size=1) +
    scale_fill_manual(values = colors) +
    facet_grid(rows = vars(Day), cols = vars(Subset), scales = "free_y") +
    scale_y_log10() + 
    stat_pvalue_manual(
        stat.test,
        label = "fc_label",
        tip.length = 0.01,
        size = 2.2,
        bracket.size = 0.3,
        inherit.aes = FALSE
    ) +
    theme + theme(strip.text = element_blank())   # Remove facet strip text

In [12]:
# Widened from 4x4 to fit fold-change bracket labels (longer than plain "*"/"**" stars)
pdf("2024-009-Fig01i.pdf", width = 7.5, height = 6.5)
print(p)
dev.off()

pdf 
  2

## Day 8 Lo / Non-Term / Term Subsets

In [13]:
# Isolate Lo / Non-Term / Term subsets, Day 8 only (freq of parent, not counts)
plot_data_d8 <- subsets %>%
    filter( Subset %in% c("Lo", "Non-Term", "Term"),
            Day == "D8.0",
            is.na(Kaech_in_Gerlach),
            !(Genotype == "KO" & Transduction == "Runx3")
    ) %>%
    select(-Kaech_in_Gerlach) %>%
    mutate(Treatment = case_when(
        Genotype == "KO" & Transduction == "mock" ~ "KO",
        Genotype == "WT" & Transduction == "mock" ~ "WT",
        Genotype == "WT" & Transduction == "Runx3" ~ "OE",
        TRUE ~ NA_character_
    )) %>%
    mutate( Treatment = factor(Treatment, levels = c("KO", "WT", "OE")),
            Subset = factor(Subset, levels = c("Lo", "Non-Term", "Term"))
    )

# BH-adjusted pairwise comparisons vs WT (matches the WT-referenced direction
# used above), adjusted within each Subset panel
stat.test_d8 <- plot_padj(
    data            = plot_data_d8,
    comparisons     = list(c("WT", "KO"), c("WT", "OE")),
    p_adjust_method = "BH",
    y_col           = "Percent",
    x_col           = "Treatment",
    facet_col       = "Subset",
    adjust_scope    = "intrafacet",
    paired          = FALSE
)

# Export full comparison stats (p, p.adj, fold changes, effect sizes) for reporting
write.csv(stat.test_d8, "2024-009-Fig01j-percent.csv", row.names = FALSE)

# Bracket labels: fold change plus significance (blank when not significant)
# Percent is plotted on a linear scale (bounded 0-100%), so unlike the Number
# plots above, y.position does not need a log10 transform
stat.test_d8 <- stat.test_d8 %>%
    mutate(
        fc_label = paste0(format_fc(foldmean), "x", ifelse(p.adj.signif == "ns", "", p.adj.signif))
    )

p_d8 <- ggplot(plot_data_d8, aes(x = Treatment, y = Percent, fill = Treatment)) +
    geom_boxplot(outliers = FALSE) +
    geom_point(size = 1) +
    scale_fill_manual(values = colors) +
    facet_grid(cols = vars(Subset), scales = "free_y") +
    stat_pvalue_manual(
        stat.test_d8,
        label = "fc_label",
        tip.length = 0.01,
        size = 2.2,
        bracket.size = 0.3,
        inherit.aes = FALSE
    ) +
    theme

In [14]:
pdf("2024-009-Fig01j-percent.pdf", width = 6, height = 4)
print(p_d8)
dev.off()

pdf 
  2

### Numbers (raw counts)

In [15]:
# Same Lo / Non-Term / Term subsets, Day 8, but as raw counts (Number) rather than freq of parent
plot_data_d8_num <- total_numbers_norm %>%
    filter( Subset %in% c("Lo", "Non-Term", "Term"),
            Day == "D8.0",
            is.na(Kaech_in_Gerlach),
            !(Genotype == "KO" & Transduction == "Runx3")
    ) %>%
    select(-Kaech_in_Gerlach) %>%
    mutate(Treatment = case_when(
        Genotype == "KO" & Transduction == "mock" ~ "KO",
        Genotype == "WT" & Transduction == "mock" ~ "WT",
        Genotype == "WT" & Transduction == "Runx3" ~ "OE",
        TRUE ~ NA_character_
    )) %>%
    mutate( Treatment = factor(Treatment, levels = c("KO", "WT", "OE")),
            Subset = factor(Subset, levels = c("Lo", "Non-Term", "Term"))
    )

# BH-adjusted pairwise comparisons vs WT, adjusted within each Subset panel
stat.test_d8_num <- plot_padj(
    data            = plot_data_d8_num,
    comparisons     = list(c("WT", "KO"), c("WT", "OE")),
    p_adjust_method = "BH",
    y_col           = "Number",
    x_col           = "Treatment",
    facet_col       = "Subset",
    adjust_scope    = "intrafacet",
    paired          = FALSE
)

# Export full comparison stats (p, p.adj, fold changes, effect sizes) for reporting
write.csv(stat.test_d8_num, "2024-009-Fig01j-numbers.csv", row.names = FALSE)

# Bracket labels: fold change plus significance (blank when not significant)
format_fc <- function(x) ifelse(abs(x) < 1, signif(x, 1), round(x, 1))
stat.test_d8_num <- stat.test_d8_num %>%
    mutate(
        fc_label = paste0(format_fc(foldmean), "x", ifelse(p.adj.signif == "ns", "", p.adj.signif)),
        # geom_bracket()'s y.position aesthetic bypasses scale_y_log10()'s data
        # transform, so it must be supplied already in log10 units
        y.position = log10(y.position)
    )

p_d8_num <- ggplot(plot_data_d8_num, aes(x = Treatment, y = Number, fill = Treatment)) +
    geom_boxplot(outliers = FALSE) +
    geom_point(size = 1) +
    scale_fill_manual(values = colors) +
    facet_grid(cols = vars(Subset), scales = "free_y") +
    scale_y_log10() +
    stat_pvalue_manual(
        stat.test_d8_num,
        label = "fc_label",
        tip.length = 0.01,
        size = 2.2,
        bracket.size = 0.3,
        inherit.aes = FALSE
    ) +
    theme

In [16]:
pdf("2024-009-Fig01j-numbers.pdf", width = 6, height = 4)
print(p_d8_num)
dev.off()

pdf 
  2